# E5 — Risk–Coverage, AURC, and Stress-Recall Analysis
## Calib-BERT v2 Reconstruction

This notebook consumes the **frozen E2 prediction-level outputs**. It does **not** retrain BERT.

### Primary scientific question
Do calibration interventions improve the usefulness of model confidence for **selective prediction / abstention**, not merely probability calibration metrics?

### Primary E5 conditions
These three conditions are predeclared from the E2 design:
1. **M0 Vanilla BERT — uncalibrated**
2. **M0 Vanilla BERT + Temperature Scaling**
3. **M1 LS ε=0.10 — uncalibrated**

Platt, Beta, and Isotonic are retained as **secondary/sensitivity analyses**, not used to redefine the primary comparison after seeing official-test outcomes.

---

## Important interpretation

E5 intentionally separates two concepts:

### A. Rank-based selective prediction
Examples:
- risk–coverage curve
- AURC
- risk @ 90/80/70/50% coverage

These metrics depend on the **ranking** of confidence values.

A positive scalar temperature scaling preserves logit ordering. Therefore, for M0, **Temperature Scaling may improve ECE/Brier/NLL while leaving the rank-based risk–coverage curve unchanged**. That is not a failure: it demonstrates that *probability calibration* and *selective ranking* are different properties.

### B. Fixed-threshold selective prediction
Examples:
- coverage at confidence ≥ 0.80 / 0.90 / 0.95
- selective error at those thresholds
- HCE counts

These metrics depend on the **absolute scale** of confidence. Temperature scaling can change them even when the rank ordering is unchanged.

---

## E5 is descriptive
Formal cluster-aware confidence intervals and multiplicity-controlled paired inference are deferred to **E7**. E5 saves the required prediction-level summaries for that later analysis.


In [ ]:
# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

# E2 default output directory in Colab.
E2_DEFAULT_DIR = Path("/content/calibbert_v2_e2/outputs_e2")

# E5 output directory.
E5_OUT_DIR = Path("/content/calibbert_v2_e2/outputs_e5")
E5_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate locations searched automatically.
SEARCH_DIRS = [
    E2_DEFAULT_DIR,
    Path("/content"),
    Path("."),
]

PREDICTION_FILENAME = "e2_prediction_level_long.csv"
LOCK_FILENAMES = ["E2_CALIBRATION_LOCK.json", "E2_CALIBRATION_LOCK(1).json"]
E2_BENCHMARK_FILENAME = "e2_official_test_calibration_benchmark.csv"

EXPECTED_N = 715
FIXED_COVERAGES = [0.90, 0.80, 0.70, 0.50]
FIXED_THRESHOLDS = [0.80, 0.90, 0.95]


## 1. Imports and input auto-discovery

In [ ]:
import json
import math
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_first(filename_options, dirs):
    if isinstance(filename_options, str):
        filename_options = [filename_options]
    for d in dirs:
        for fn in filename_options:
            p = Path(d) / fn
            if p.exists():
                return p
    return None

PRED_PATH = find_first(PREDICTION_FILENAME, SEARCH_DIRS)
LOCK_PATH = find_first(LOCK_FILENAMES, SEARCH_DIRS)
BENCHMARK_PATH = find_first(E2_BENCHMARK_FILENAME, SEARCH_DIRS)

if PRED_PATH is None:
    raise FileNotFoundError(
        "e2_prediction_level_long.csv was not found. "
        "Open the completed E2 notebook and rerun the official-test benchmark/export cell, "
        "then place the file in /content/calibbert_v2_e2/outputs_e2/ or /content/."
    )

print("Prediction file :", PRED_PATH)
print("E2 lock         :", LOCK_PATH)
print("E2 benchmark    :", BENCHMARK_PATH)

pred = pd.read_csv(PRED_PATH)
print("\nPrediction rows:", len(pred))
print("Columns:", pred.columns.tolist())


## 2. Verify E2 lock and prediction schema

The E2 lock is audited, not rewritten.  
This preserves the chronology that the E2 comparison set was fixed before official-test benchmarking.


In [ ]:
required_cols = {
    "row_index", "id", "post_id", "true_label",
    "model", "calibrator",
    "p_not_stress", "p_stress",
    "predicted_label", "confidence"
}
missing = required_cols - set(pred.columns)
if missing:
    raise KeyError(f"Prediction file is missing required columns: {sorted(missing)}")

if LOCK_PATH is not None:
    e2_lock = json.loads(Path(LOCK_PATH).read_text())
    print(json.dumps(e2_lock, indent=2))

    if e2_lock.get("automatic_test_based_method_selection") is not False:
        raise RuntimeError("Unexpected E2 lock: automatic_test_based_method_selection is not false.")
    if e2_lock.get("official_test_metrics_used_for_calibrator_fit") is not False:
        raise RuntimeError("Unexpected E2 lock: test metrics were apparently used for calibrator fitting.")
else:
    warnings.warn(
        "E2 lock file not found in Colab. E5 can run, but copy the lock into the final evidence package."
    )

# Each model × calibrator condition should have the same official-test cases.
condition_counts = (
    pred.groupby(["model", "calibrator"])["row_index"]
        .nunique()
        .sort_values()
)
display(condition_counts)

if condition_counts.min() != EXPECTED_N or condition_counts.max() != EXPECTED_N:
    raise RuntimeError(
        f"Expected {EXPECTED_N} unique test cases per condition; got range "
        f"{condition_counts.min()}–{condition_counts.max()}."
    )

# Verify that labels and IDs align across all conditions.
base_keys = (
    pred[["row_index", "id", "post_id", "true_label"]]
    .drop_duplicates()
    .sort_values("row_index")
    .reset_index(drop=True)
)
if len(base_keys) != EXPECTED_N:
    raise RuntimeError(f"Expected {EXPECTED_N} unique official-test rows; found {len(base_keys)}.")

print("E2 prediction alignment: PASS")


## 3. Freeze E5 primary and secondary conditions

Primary conditions are theory-driven:
- uncalibrated baseline,
- standard order-preserving Temperature Scaling,
- validation-selected training-time LS ε=0.10.

No E5 condition is chosen because it has the lowest official-test error.


In [ ]:
PRIMARY_CONDITIONS = [
    ("M0_Vanilla_BERT", "none", "M0 Vanilla"),
    ("M0_Vanilla_BERT", "temperature", "M0 + Temperature Scaling"),
    ("M1_LS_0.10", "none", "M1 LS 0.10"),
]

SECONDARY_CALIBRATORS = ["platt", "beta", "isotonic"]

available = set(zip(pred["model"], pred["calibrator"]))

for model, calibrator, label in PRIMARY_CONDITIONS:
    if (model, calibrator) not in available:
        raise RuntimeError(f"Missing primary condition: {(model, calibrator)}")

e5_lock = {
    "analysis": "E5 Risk-Coverage and AURC",
    "source_prediction_file": str(PRED_PATH),
    "n_official_test": EXPECTED_N,
    "primary_conditions": [
        {"model": m, "calibrator": c, "label": lab}
        for m, c, lab in PRIMARY_CONDITIONS
    ],
    "secondary_calibrators": SECONDARY_CALIBRATORS,
    "fixed_coverages": FIXED_COVERAGES,
    "fixed_thresholds": FIXED_THRESHOLDS,
    "aurc_definition": "mean cumulative risk over top-k confidence-ranked predictions, k=1..N",
    "stress_recall_definition": (
        "correctly predicted stress examples retained at coverage k / total true stress examples in full test set"
    ),
    "test_based_condition_selection": False,
    "formal_cluster_bootstrap_deferred_to": "E7",
}

(E5_OUT_DIR / "E5_ANALYSIS_LOCK.json").write_text(json.dumps(e5_lock, indent=2))
print(json.dumps(e5_lock, indent=2))


## 4. Core selective-prediction functions

For a model condition:

1. Sort cases by descending confidence.
2. At each `k = 1..N`, retain the top-`k` cases.
3. Coverage is `k/N`.
4. Risk is the classification error among retained cases.
5. **AURC** is the discrete mean of risk over all coverage points.

### Stress recall under abstention
At each coverage:
\[
\text{StressRecallRetained}(k)
=
\frac{\#\{\text{true Stress, predicted Stress, retained}\}}
{\#\{\text{true Stress in full test set}\}}
\]

This deliberately penalizes abstention on true Stress cases.


In [ ]:
def condition_frame(df, model, calibrator):
    x = df[
        (df["model"] == model) &
        (df["calibrator"] == calibrator)
    ].copy()

    x = x.sort_values("row_index").reset_index(drop=True)

    if len(x) != EXPECTED_N:
        raise ValueError(f"{model}/{calibrator}: expected {EXPECTED_N} rows, got {len(x)}")

    # Recompute correctness rather than trusting any precomputed field.
    x["correct"] = (x["predicted_label"].astype(int) == x["true_label"].astype(int)).astype(int)
    x["error"] = 1 - x["correct"]
    return x

def risk_coverage_curve(x):
    # Stable tie handling: confidence descending, then row_index ascending.
    s = x.sort_values(
        ["confidence", "row_index"],
        ascending=[False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    n = len(s)
    k = np.arange(1, n + 1)
    coverage = k / n
    cum_errors = np.cumsum(s["error"].to_numpy())
    risk = cum_errors / k

    true_stress = (s["true_label"].to_numpy() == 1)
    pred_stress = (s["predicted_label"].to_numpy() == 1)
    true_stress_total = int(true_stress.sum())

    tp_retained = np.cumsum((true_stress & pred_stress).astype(int))
    stress_recall_retained = tp_retained / true_stress_total

    # Conditional stress recall among covered true-stress cases.
    stress_covered = np.cumsum(true_stress.astype(int))
    conditional_stress_recall = np.divide(
        tp_retained,
        stress_covered,
        out=np.full_like(tp_retained, np.nan, dtype=float),
        where=stress_covered > 0,
    )

    curve = pd.DataFrame({
        "rank_k": k,
        "coverage": coverage,
        "risk": risk,
        "accuracy_retained": 1.0 - risk,
        "stress_recall_retained_global": stress_recall_retained,
        "stress_recall_conditional_covered": conditional_stress_recall,
        "threshold_at_k": s["confidence"].to_numpy(),
    })

    aurc = float(np.mean(risk))
    return curve, aurc

def exact_topk_summary(curve, requested_coverage):
    n = len(curve)
    # Nearest integer top-k, at least 1.
    k = max(1, min(n, int(round(requested_coverage * n))))
    row = curve.iloc[k - 1]
    return {
        "requested_coverage": requested_coverage,
        "actual_coverage": float(row["coverage"]),
        "k_retained": int(row["rank_k"]),
        "confidence_cutoff": float(row["threshold_at_k"]),
        "risk": float(row["risk"]),
        "accuracy_retained": float(row["accuracy_retained"]),
        "stress_recall_retained_global": float(row["stress_recall_retained_global"]),
        "stress_recall_conditional_covered": float(row["stress_recall_conditional_covered"]),
    }

def threshold_summary(x, tau):
    conf = x["confidence"].to_numpy(float)
    y = x["true_label"].to_numpy(int)
    pred_y = x["predicted_label"].to_numpy(int)

    keep = conf >= tau
    n_keep = int(keep.sum())
    coverage = n_keep / len(x)

    if n_keep == 0:
        selective_error = np.nan
        retained_accuracy = np.nan
    else:
        selective_error = float(np.mean(pred_y[keep] != y[keep]))
        retained_accuracy = 1.0 - selective_error

    hce = int(((pred_y != y) & keep).sum())

    total_true_stress = int((y == 1).sum())
    tp_retained = int(((y == 1) & (pred_y == 1) & keep).sum())
    stress_recall_retained_global = tp_retained / total_true_stress

    covered_true_stress = int(((y == 1) & keep).sum())
    conditional_stress_recall = (
        tp_retained / covered_true_stress
        if covered_true_stress > 0 else np.nan
    )

    return {
        "threshold": tau,
        "coverage": coverage,
        "n_retained": n_keep,
        "selective_error": selective_error,
        "accuracy_retained": retained_accuracy,
        "hce": hce,
        "stress_recall_retained_global": stress_recall_retained_global,
        "stress_recall_conditional_covered": conditional_stress_recall,
    }


## 5. Compute primary risk–coverage curves and AURC

In [ ]:
primary_curves = {}
primary_aurc_rows = []
primary_fixed_coverage_rows = []
primary_threshold_rows = []

for model, calibrator, label in PRIMARY_CONDITIONS:
    x = condition_frame(pred, model, calibrator)
    curve, aurc = risk_coverage_curve(x)

    curve["model"] = model
    curve["calibrator"] = calibrator
    curve["label"] = label
    primary_curves[label] = curve

    primary_aurc_rows.append({
        "model": model,
        "calibrator": calibrator,
        "label": label,
        "aurc": aurc,
        "full_coverage_risk": float(curve.iloc[-1]["risk"]),
        "full_coverage_accuracy": float(curve.iloc[-1]["accuracy_retained"]),
        "full_coverage_stress_recall": float(curve.iloc[-1]["stress_recall_retained_global"]),
    })

    for c in FIXED_COVERAGES:
        row = exact_topk_summary(curve, c)
        primary_fixed_coverage_rows.append({
            "model": model,
            "calibrator": calibrator,
            "label": label,
            **row,
        })

    for tau in FIXED_THRESHOLDS:
        row = threshold_summary(x, tau)
        primary_threshold_rows.append({
            "model": model,
            "calibrator": calibrator,
            "label": label,
            **row,
        })

primary_aurc = pd.DataFrame(primary_aurc_rows)
primary_fixed_coverage = pd.DataFrame(primary_fixed_coverage_rows)
primary_thresholds = pd.DataFrame(primary_threshold_rows)

all_primary_curves = pd.concat(primary_curves.values(), ignore_index=True)

primary_aurc.to_csv(E5_OUT_DIR / "e5_primary_aurc.csv", index=False)
primary_fixed_coverage.to_csv(E5_OUT_DIR / "e5_primary_fixed_coverage.csv", index=False)
primary_thresholds.to_csv(E5_OUT_DIR / "e5_primary_fixed_thresholds.csv", index=False)
all_primary_curves.to_csv(E5_OUT_DIR / "e5_primary_risk_coverage_curves.csv", index=False)

display(primary_aurc)
display(primary_fixed_coverage)
display(primary_thresholds)


## 6. Explicitly test the expected M0 vs M0+TS rank invariance

Temperature Scaling with a positive scalar should preserve the confidence ranking for the same M0 logits.  
If ranking is identical, rank-based risk–coverage and AURC should also be identical up to numerical precision.


In [ ]:
m0 = condition_frame(pred, "M0_Vanilla_BERT", "none")
m0_ts = condition_frame(pred, "M0_Vanilla_BERT", "temperature")

rank_m0 = (
    m0.sort_values(["confidence", "row_index"], ascending=[False, True], kind="mergesort")
      ["row_index"].to_numpy()
)
rank_ts = (
    m0_ts.sort_values(["confidence", "row_index"], ascending=[False, True], kind="mergesort")
         ["row_index"].to_numpy()
)

same_ranking = bool(np.array_equal(rank_m0, rank_ts))

curve_m0 = primary_curves["M0 Vanilla"]
curve_ts = primary_curves["M0 + Temperature Scaling"]

max_risk_curve_diff = float(np.max(np.abs(curve_m0["risk"].values - curve_ts["risk"].values)))
aurc_m0 = float(primary_aurc.loc[primary_aurc["label"]=="M0 Vanilla", "aurc"].iloc[0])
aurc_ts = float(primary_aurc.loc[primary_aurc["label"]=="M0 + Temperature Scaling", "aurc"].iloc[0])

rank_invariance = {
    "same_row_ranking": same_ranking,
    "max_absolute_risk_curve_difference": max_risk_curve_diff,
    "aurc_m0": aurc_m0,
    "aurc_m0_temperature": aurc_ts,
    "absolute_aurc_difference": abs(aurc_m0 - aurc_ts),
}

(E5_OUT_DIR / "e5_temperature_rank_invariance.json").write_text(
    json.dumps(rank_invariance, indent=2)
)
print(json.dumps(rank_invariance, indent=2))


## 7. Primary Figure A — Risk–Coverage curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for label, curve in primary_curves.items():
    ax.plot(curve["coverage"], curve["risk"], label=label)

ax.set_xlabel("Coverage")
ax.set_ylabel("Selective risk")
ax.set_title("Risk–Coverage Curves — Primary E5 Conditions")
ax.set_xlim(0, 1)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()

fig.savefig(E5_OUT_DIR / "Figure_E5_Risk_Coverage.png", dpi=300, bbox_inches="tight")
fig.savefig(E5_OUT_DIR / "Figure_E5_Risk_Coverage.pdf", bbox_inches="tight")
plt.show()


## 8. Primary Figure B — Stress-class recall retained versus coverage

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for label, curve in primary_curves.items():
    ax.plot(
        curve["coverage"],
        curve["stress_recall_retained_global"],
        label=label
    )

ax.set_xlabel("Coverage")
ax.set_ylabel("Retained Stress-class recall")
ax.set_title("Stress Recall Under Selective Prediction")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()

fig.savefig(E5_OUT_DIR / "Figure_E5_Stress_Recall_Coverage.png", dpi=300, bbox_inches="tight")
fig.savefig(E5_OUT_DIR / "Figure_E5_Stress_Recall_Coverage.pdf", bbox_inches="tight")
plt.show()


## 9. Secondary/sensitivity AURC analysis

All E2 conditions are summarized here.  
This table is **secondary evidence** and must not be used to retroactively redefine the primary E5 comparison.


In [ ]:
secondary_rows = []

for (model, calibrator), x0 in pred.groupby(["model", "calibrator"]):
    x = x0.sort_values("row_index").reset_index(drop=True).copy()
    x["correct"] = (x["predicted_label"].astype(int) == x["true_label"].astype(int)).astype(int)
    x["error"] = 1 - x["correct"]

    curve, aurc = risk_coverage_curve(x)

    secondary_rows.append({
        "model": model,
        "calibrator": calibrator,
        "aurc": aurc,
        "full_coverage_risk": float(curve.iloc[-1]["risk"]),
        "full_coverage_accuracy": float(curve.iloc[-1]["accuracy_retained"]),
        "full_coverage_stress_recall": float(curve.iloc[-1]["stress_recall_retained_global"]),
    })

secondary_aurc = pd.DataFrame(secondary_rows).sort_values(["model", "calibrator"])
secondary_aurc.to_csv(E5_OUT_DIR / "e5_all_conditions_aurc_sensitivity.csv", index=False)
display(secondary_aurc)


## 10. Publication table for E5

The main table combines:
- AURC
- risk at fixed **coverage**
- Stress recall at fixed **coverage**

Threshold-based results remain a separate table because calibrated probability thresholds and rank-based coverage answer different questions.


In [ ]:
wide_rows = []

for _, a in primary_aurc.iterrows():
    row = {
        "model": a["model"],
        "calibrator": a["calibrator"],
        "label": a["label"],
        "AURC": a["aurc"],
        "full_coverage_risk": a["full_coverage_risk"],
        "full_coverage_stress_recall": a["full_coverage_stress_recall"],
    }

    sub = primary_fixed_coverage[primary_fixed_coverage["label"] == a["label"]]
    for _, r in sub.iterrows():
        c = int(round(r["requested_coverage"] * 100))
        row[f"risk_at_{c}pct_coverage"] = r["risk"]
        row[f"stress_recall_at_{c}pct_coverage"] = r["stress_recall_retained_global"]
        row[f"confidence_cutoff_for_{c}pct_coverage"] = r["confidence_cutoff"]

    wide_rows.append(row)

publication_table = pd.DataFrame(wide_rows)
publication_table.to_csv(E5_OUT_DIR / "Table_E5_Risk_Coverage_AURC.csv", index=False)

threshold_table = primary_thresholds.copy()
threshold_table.to_csv(E5_OUT_DIR / "Table_E5_Fixed_Threshold_Selectivity.csv", index=False)

display(publication_table)
display(threshold_table)


## 11. Export prediction-level ranking information for E7

E7 will use `post_id`-cluster bootstrap.  
This file preserves each primary condition's confidence ranking and correctness without rerunning BERT.


In [ ]:
e7_exports = []

for model, calibrator, label in PRIMARY_CONDITIONS:
    x = condition_frame(pred, model, calibrator)
    x = x.sort_values(
        ["confidence", "row_index"],
        ascending=[False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    x["confidence_rank"] = np.arange(1, len(x)+1)
    x["coverage_at_rank"] = x["confidence_rank"] / len(x)
    x["label_condition"] = label

    e7_exports.append(
        x[[
            "row_index", "id", "post_id", "true_label",
            "model", "calibrator", "label_condition",
            "p_not_stress", "p_stress",
            "predicted_label", "confidence",
            "correct", "error",
            "confidence_rank", "coverage_at_rank"
        ]]
    )

e7_input = pd.concat(e7_exports, ignore_index=True)
e7_input.to_csv(E5_OUT_DIR / "e5_primary_prediction_ranking_for_E7.csv", index=False)

print("Saved E7 input:", E5_OUT_DIR / "e5_primary_prediction_ranking_for_E7.csv")
print("Rows:", len(e7_input))


## 12. E5 completion gate

E5 can be frozen when all checks below are satisfied:

- [ ] `e2_prediction_level_long.csv` contains 715 aligned official-test cases for every condition
- [ ] the historical E2 lock is available in the evidence package
- [ ] `E5_ANALYSIS_LOCK.json` is written before interpreting E5 results
- [ ] primary conditions are exactly:
  - M0 Vanilla
  - M0 + Temperature Scaling
  - M1 LS 0.10
- [ ] AURC uses confidence ranking, not arbitrary probability thresholds
- [ ] fixed-coverage and fixed-threshold analyses are reported separately
- [ ] Stress-class recall under abstention is reported
- [ ] Platt/Beta/Isotonic remain sensitivity analyses
- [ ] no E5 condition is promoted to “best” based only on official-test results
- [ ] `e5_primary_prediction_ranking_for_E7.csv` is preserved for E7 cluster-aware inference

### Required E5 files
- `E5_ANALYSIS_LOCK.json`
- `e5_primary_aurc.csv`
- `e5_primary_fixed_coverage.csv`
- `e5_primary_fixed_thresholds.csv`
- `e5_primary_risk_coverage_curves.csv`
- `e5_temperature_rank_invariance.json`
- `Table_E5_Risk_Coverage_AURC.csv`
- `Table_E5_Fixed_Threshold_Selectivity.csv`
- `Figure_E5_Risk_Coverage.png/pdf`
- `Figure_E5_Stress_Recall_Coverage.png/pdf`
- `e5_primary_prediction_ranking_for_E7.csv`

After E5 is frozen, the next major experiment is **E3 Cross-Backbone Replication**.  
E7 statistical inference can also be run once E3/E4 comparison structures are finalized.
